# GreenLight: Интерактивное руководство по модели теплицы

Этот блокнот поможет разобраться, как устроена модель **GreenLight** — платформа для моделирования динамики тепличных систем.

## Что моделирует GreenLight?

Модель описывает физические процессы в теплице через систему **обыкновенных дифференциальных уравнений (ОДУ)**:

| Категория | Что моделируется |
|-----------|-----------------|
| **Климат теплицы** | Температуры (воздух, покрытие, экраны, трубы, пол, растения) |
| **Влажность** | Давление водяного пара внутри и снаружи |
| **CO₂** | Концентрация углекислого газа |
| **Освещение** | Солнечная радиация (PAR, NIR, FIR) + лампы (HPS/LED) |
| **Рост культуры** | Фотосинтез, распределение углеводов по органам |
| **Управление** | Отопление, вентиляция, экраны, подсветка |

Основан на работах **Vanthoor (2011)** и **Katzin (2021)**.

In [ ]:
# Установка зависимостей (раскомментируйте при необходимости)
# !pip install numpy numexpr scipy pandas matplotlib

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path, PurePath

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.grid"] = True

MODELS_DIR = Path("../greenlight/models/katzin_2021/definition")
print("Модели загружены из:", MODELS_DIR.resolve())

---
## 1. Архитектура модели: из чего она собирается

Модель Katzin 2021 — это **составная** модель, собранная из нескольких JSON-файлов. Каждый файл описывает подсистему теплицы.

Посмотрим файл `main_katzin_2021.json`, который задаёт **порядок сборки**:

In [ ]:
# Читаем главный файл модели
with open(MODELS_DIR / "main_katzin_2021.json", "r") as f:
    main_model = json.load(f)

# Показываем порядок обработки
print("=== Порядок сборки модели (processing_order) ===\n")
for i, item in enumerate(main_model["processing_order"], 1):
    if isinstance(item, str):
        print(f"  {i}. 📄 {item}")
    elif isinstance(item, dict):
        print(f"  {i}. ⚙️  Настройки: {json.dumps(item, indent=6)}")

print("\n=== Описание модели ===")
print(main_model.get("About", "Нет описания"))

Каждый JSON содержит переменные с полями:
- `definition` — математическая формула
- `type` — тип: `state` (ОДУ), `aux` (вспомогательная), `const` (константа), `input` (вход), `function` (функция)
- `unit` — единица измерения
- `init` — начальное значение (для state)

---
## 2. Загружаем и анализируем все переменные модели

Соберём все переменные из всех JSON-файлов и классифицируем их:

In [ ]:
def load_all_variables(models_dir, main_file="main_katzin_2021.json"):
    """Рекурсивно загружает все переменные из JSON-файлов модели."""
    all_vars = {}

    def process_json(filepath):
        with open(filepath, "r") as f:
            data = json.load(f)

        # Если есть processing_order — рекурсивно обрабатываем
        if "processing_order" in data:
            base = filepath.parent
            for item in data["processing_order"]:
                if isinstance(item, str) and item.endswith(".json"):
                    process_json(base / item)
                elif isinstance(item, dict):
                    extract_vars(item, str(filepath.name))

        extract_vars(data, str(filepath.name))

    def extract_vars(data, source):
        for section_name, section in data.items():
            if not isinstance(section, dict):
                continue
            for var_name, var_def in section.items():
                if isinstance(var_def, dict) and "definition" in var_def:
                    all_vars[var_name] = {
                        "definition": var_def["definition"],
                        "type": var_def.get("type", "unknown"),
                        "unit": var_def.get("unit", ""),
                        "description": var_def.get("description", ""),
                        "reference": var_def.get("reference", ""),
                        "init": var_def.get("init", None),
                        "source": source,
                    }

    process_json(models_dir / main_file)
    return all_vars

all_vars = load_all_variables(MODELS_DIR)

# Классификация
by_type = {}
for name, v in all_vars.items():
    t = v["type"]
    by_type.setdefault(t, []).append(name)

print(f"Всего переменных: {len(all_vars)}\n")
for t, names in sorted(by_type.items(), key=lambda x: -len(x[1])):
    print(f"  {t:12s}: {len(names):4d} переменных")


---
## 3. Переменные состояния (States) — сердце модели

Переменные состояния — это те, для которых записаны дифференциальные уравнения `dx/dt = f(...)`. Именно они «живут» во времени и описывают динамику теплицы:

In [ ]:
# Показываем все переменные состояния с описаниями и начальными значениями
states = {k: v for k, v in all_vars.items() if v["type"] == "state"}

print(f"{'Переменная':<20} {'Нач. знач.':<12} {'Ед. изм.':<20} {'Описание'}")
print("=" * 100)
for name, v in sorted(states.items()):
    init = v["init"] if v["init"] else "—"
    desc = v["description"][:50] if v["description"] else "—"
    print(f"{name:<20} {str(init):<12} {v['unit']:<20} {desc}")

---
## 4. Граф зависимостей: кто от кого зависит

Построим граф зависимостей переменных. Каждая переменная в своей `definition` ссылается на другие переменные:

In [ ]:
import re

def find_dependencies(definition, known_vars):
    """Находит все переменные, от которых зависит данное определение."""
    deps = set()
    # Ищем идентификаторы (слова), которые являются известными переменными
    tokens = set(re.findall(r'\b[a-zA-Z_]\w*\b', definition))
    # Исключаем математические функции и ключевые слова
    math_funcs = {'exp', 'log', 'sqrt', 'sin', 'cos', 'tan', 'abs', 'max', 'min',
                  'floor', 'ceil', 'sinh', 'cosh', 'tanh', 'where', 'pi', 'inf'}
    for token in tokens:
        if token in known_vars and token not in math_funcs:
            deps.add(token)
    return deps

# Строим граф зависимостей
dep_graph = {}
for name, v in all_vars.items():
    dep_graph[name] = find_dependencies(v["definition"], all_vars)

# Для каждой переменной состояния: покажем дерево зависимостей (1 уровень)
print("=== Прямые зависимости переменных состояния ===\n")
for state_name in sorted(states.keys()):
    deps = dep_graph.get(state_name, set())
    deps_by_type = {}
    for d in deps:
        t = all_vars[d]["type"] if d in all_vars else "?"
        deps_by_type.setdefault(t, []).append(d)
    
    print(f"📊 {state_name} (dx/dt зависит от {len(deps)} переменных):")
    for t, names in sorted(deps_by_type.items()):
        print(f"   [{t}]: {', '.join(sorted(names))}")
    print()

---
## 5. Подробный разбор подсистем

### 5.1 Тепловой баланс: температура воздуха (`tAir`)

Температура воздуха — ключевая переменная. Посмотрим, какие потоки энергии на неё влияют:

In [ ]:
def show_equation(var_name, all_vars, depth=0, max_depth=1):
    """Рекурсивно показывает уравнение переменной и её зависимости."""
    v = all_vars.get(var_name)
    if not v:
        return
    
    indent = "  " * depth
    desc = v["description"][:60] if v["description"] else ""
    print(f"{indent}{'─' if depth > 0 else '═'}─ {var_name} [{v['type']}] = {v['definition'][:100]}")
    if desc:
        print(f"{indent}   ({desc})")
    if v["unit"]:
        print(f"{indent}   Единицы: {v['unit']}")
    
    if depth < max_depth:
        deps = find_dependencies(v["definition"], all_vars)
        for dep in sorted(deps):
            if all_vars.get(dep, {}).get("type") == "aux":
                show_equation(dep, all_vars, depth + 1, max_depth)

print("═══ Уравнение для tAir (температура воздуха в теплице) ═══\n")
show_equation("tAir", all_vars, max_depth=1)

print("\n\n═══ Уравнение для tCan (температура растений) ═══\n")
show_equation("tCan", all_vars, max_depth=1)

### 5.2 Освещение: PAR, NIR, FIR

Радиация разделена на 3 спектральных диапазона:
- **PAR** (400-700 нм) — фотосинтетически активная
- **NIR** (700-2500 нм) — ближний инфракрасный (нагрев)
- **FIR** (>2500 нм) — дальний инфракрасный (тепловое излучение)

Посмотрим все переменные, связанные с освещением:

In [ ]:
# Классификация переменных освещения
light_keywords = ['Par', 'Nir', 'Fir', 'Lamp', 'lamp', 'iGlob', 'Rad', 'rad']
light_vars = {k: v for k, v in all_vars.items()
              if any(kw in k for kw in light_keywords)}

# Группируем по источнику
solar_vars = {k: v for k, v in light_vars.items() if 'Sun' in k or 'iGlob' in k}
lamp_vars = {k: v for k, v in light_vars.items() if 'Lamp' in k or 'lamp' in k}
fir_vars = {k: v for k, v in light_vars.items() if 'Fir' in k or 'fir' in k}

print(f"Всего переменных освещения: {len(light_vars)}")
print(f"  Солнечная радиация:  {len(solar_vars)}")
print(f"  Лампы (HPS/LED):     {len(lamp_vars)}")
print(f"  FIR (тепловое):      {len(fir_vars)}")

print("\n─── Ключевые переменные ламп ───")
for name in sorted(lamp_vars):
    v = lamp_vars[name]
    print(f"  {name:<30} [{v['type']:<5}] {v['unit']:<15} = {v['definition'][:70]}")

### 5.3 Фотосинтез и рост культуры

Модель культуры (Vanthoor 2011, глава 9) описывает фотосинтез и распределение углеводов. Ключевое уравнение — **электронный транспорт** (`j`), зависящий от `parCan` (PAR, поглощённый растениями):

In [ ]:
# Цепочка фотосинтеза: parCan → j → p → mcAirBuf
photosynthesis_chain = ["parCan", "j", "p", "r", "mcAirBuf"]

print("═══ Цепочка фотосинтеза: свет → углеводы ═══\n")
for var in photosynthesis_chain:
    if var in all_vars:
        v = all_vars[var]
        print(f"▸ {var} [{v['type']}]")
        print(f"  Формула: {v['definition'][:120]}")
        if v["description"]:
            print(f"  Описание: {v['description'][:80]}")
        print(f"  Единицы: {v['unit']}")
        print()

# Показываем переменные роста
crop_states = [k for k in states if k.startswith("cBuf") or k.startswith("cFruit") 
               or k.startswith("cLeaf") or k.startswith("cStem") or k == "lai"]
print("─── Состояния роста культуры ───")
for name in sorted(crop_states):
    v = all_vars[name]
    desc = v["description"][:60] if v["description"] else ""
    print(f"  {name:<20} init={v['init']:<10} {v['unit']:<20} {desc}")

### 5.4 Система управления

Модель включает алгоритмы управления: вентиляция, отопление, освещение, экраны. Посмотрим управляющие переменные:

In [ ]:
# Управляющие переменные (начинаются с "u")
control_vars = {k: v for k, v in all_vars.items() if k.startswith("u") and v["type"] == "aux"}

print("═══ Управляющие переменные ═══\n")
for name in sorted(control_vars):
    v = control_vars[name]
    desc = v["description"][:70] if v["description"] else ""
    print(f"  {name:<20} {desc}")
    print(f"  {'':20} = {v['definition'][:90]}")
    print()

---
## 6. Генерация тестовых погодных данных и запуск симуляции

Для запуска модели нужны входные данные: погода за пределами теплицы. Сгенерируем синтетические данные, имитирующие осенний день в Нидерландах:

In [ ]:
def generate_weather_csv(filepath, n_days=2, dt_seconds=300):
    """Генерирует синтетические погодные данные для GreenLight.
    
    Имитирует осенний день (октябрь) в Нидерландах:
    - Солнечная радиация: колокол днём, 0 ночью
    - Температура: синусоидальный цикл 8-15°C
    - Влажность, ветер, CO₂
    """
    t_end = n_days * 86400
    time = np.arange(0, t_end + dt_seconds, dt_seconds)
    n = len(time)
    hours = time / 3600.0
    
    # Солнечная радиация (W/m²) — колокол с пиком ~350 W/m² в полдень
    sunrise, sunset = 7.5, 17.5  # часы
    iGlob = np.zeros(n)
    for i, h in enumerate(hours % 24):
        if sunrise < h < sunset:
            x = (h - (sunrise + sunset) / 2) / ((sunset - sunrise) / 2)
            iGlob[i] = 350 * max(0, np.cos(x * np.pi / 2) ** 1.5)
    # Добавляем немного шума
    iGlob += np.random.normal(0, 5, n)
    iGlob = np.clip(iGlob, 0, None)
    
    # Температура наружного воздуха (°C) — суточный цикл
    tOut = 11.5 + 3.5 * np.sin(2 * np.pi * (hours - 6) / 24) + np.random.normal(0, 0.3, n)
    
    # Давление водяного пара (Pa)
    vpOut = 1000 + 200 * np.sin(2 * np.pi * (hours - 3) / 24) + np.random.normal(0, 30, n)
    
    # CO₂ снаружи (mg/m³) — примерно 710-730
    co2Out = 720 + np.random.normal(0, 5, n)
    
    # Ветер (m/s)
    wind = 3.0 + 1.5 * np.sin(2 * np.pi * hours / 24) + np.abs(np.random.normal(0, 0.5, n))
    
    # Температура глубокой почвы (°C) — почти постоянная
    tSoOut = np.full(n, 12.0) + np.random.normal(0, 0.05, n)
    
    # Дневная сумма радиации (MJ/m²/день) — интегрируем
    dayRadSum = np.zeros(n)
    current_sum = 0
    for i in range(n):
        h = hours[i] % 24
        if h < 0.1 and i > 0:  # Новый день
            current_sum = 0
        current_sum += iGlob[i] * dt_seconds / 1e6  # W*s → MJ
        dayRadSum[i] = current_sum
    
    # isDay — бинарный флаг
    isDay = (iGlob > 10).astype(float)
    
    # isDaySmooth — сглаженная версия
    isDaySmooth = np.zeros(n)
    for i, h in enumerate(hours % 24):
        if h < sunrise:
            isDaySmooth[i] = 0
        elif h < sunrise + 1:
            isDaySmooth[i] = (h - sunrise)
        elif h < sunset - 1:
            isDaySmooth[i] = 1
        elif h < sunset:
            isDaySmooth[i] = sunset - h
        else:
            isDaySmooth[i] = 0
    
    # Создаём DataFrame
    df = pd.DataFrame({
        "Time": time,
        "iGlob": np.round(iGlob, 2),
        "tOut": np.round(tOut, 2),
        "vpOut": np.round(vpOut, 2),
        "co2Out": np.round(co2Out, 2),
        "wind": np.round(wind, 2),
        "tSoOut": np.round(tSoOut, 2),
        "dayRadSum": np.round(dayRadSum, 4),
        "isDay": isDay,
        "isDaySmooth": np.round(isDaySmooth, 4),
    })
    
    # Формат CSV как в GreenLight: 2 строки заголовков (описание + единицы), затем данные
    header_desc = ["Time from start of simulation", "Global radiation",
                   "Outdoor temperature", "Outdoor vapor pressure",
                   "Outdoor CO2 concentration", "Wind speed",
                   "Deep soil temperature", "Daily radiation sum",
                   "Is day (binary)", "Is day (smooth)"]
    header_units = ["s", "W m**-2", "°C", "Pa", "mg m**-3", "m s**-1",
                    "°C", "MJ m**-2 day**-1", "-", "-"]
    
    with open(filepath, "w") as f:
        f.write(",".join(df.columns) + "\n")
        f.write(",".join(header_desc) + "\n")
        f.write(",".join(header_units) + "\n")
        df.to_csv(f, index=False, header=False)
    
    return df

# Генерируем данные
weather_csv = Path("../greenlight/models/katzin_2021/input_data/test_data/Bleiswijk_from_20091020.csv")
weather_csv.parent.mkdir(parents=True, exist_ok=True)
weather_df = generate_weather_csv(weather_csv, n_days=2)

print(f"Сгенерировано {len(weather_df)} записей ({weather_df['Time'].iloc[-1]/3600:.0f} часов)")
print(f"Файл: {weather_csv.resolve()}\n")
print(weather_df.head(10))

In [ ]:
# Визуализация входных данных
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
hours = weather_df["Time"] / 3600

axes[0, 0].plot(hours, weather_df["iGlob"], color="orange")
axes[0, 0].set_title("Солнечная радиация (iGlob)")
axes[0, 0].set_ylabel("W/m²")
axes[0, 0].fill_between(hours, weather_df["iGlob"], alpha=0.3, color="orange")

axes[0, 1].plot(hours, weather_df["tOut"], color="red")
axes[0, 1].set_title("Температура снаружи (tOut)")
axes[0, 1].set_ylabel("°C")

axes[0, 2].plot(hours, weather_df["vpOut"], color="blue")
axes[0, 2].set_title("Давление водяного пара (vpOut)")
axes[0, 2].set_ylabel("Pa")

axes[1, 0].plot(hours, weather_df["co2Out"], color="green")
axes[1, 0].set_title("CO₂ снаружи (co2Out)")
axes[1, 0].set_ylabel("mg/m³")

axes[1, 1].plot(hours, weather_df["wind"], color="gray")
axes[1, 1].set_title("Скорость ветра (wind)")
axes[1, 1].set_ylabel("m/s")

axes[1, 2].plot(hours, weather_df["dayRadSum"], color="purple")
axes[1, 2].set_title("Дневная сумма радиации")
axes[1, 2].set_ylabel("MJ/m²/day")

for ax in axes.flat:
    ax.set_xlabel("Время (часы)")

plt.suptitle("Входные погодные данные (синтетические)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Запуск симуляции

Запускаем полную модель GreenLight: загрузка → решение ОДУ → результаты.

> **Время решения:** ~20-60 секунд для 2-дневной симуляции (BDF-солвер для жёсткой системы ОДУ)

In [ ]:
import time as time_module
from greenlight import GreenLight

# Создаём модель с 2-дневной симуляцией
n_days = 2
mdl = GreenLight(
    optional_prompt={"options": {"t_end": str(n_days * 86400)}}
)

# Шаг 1: Загрузка модели
print("1. Загрузка модели...")
t0 = time_module.time()
mdl.load()
print(f"   Загружено за {time_module.time() - t0:.1f} сек")
print(f"   Переменных: {len(mdl.variables)}")
print(f"   Состояний: {len(mdl.states)}")
print(f"   Констант: {len(mdl.consts)}")
print(f"   Входов: {len(mdl.inputs)}")
print(f"   Вспомогательных: {len(mdl.aux)}")
print(f"   Функций: {len(mdl.functions)}")

# Шаг 2: Решение ОДУ
print("\n2. Решение ОДУ системы...")
t0 = time_module.time()
mdl.solve()
print(f"   Решено за {time_module.time() - t0:.1f} сек")
print(f"   Размер full_sol: {mdl.full_sol.shape}")
print(f"   Временной диапазон: {mdl.full_sol['Time'].iloc[0]/3600:.1f} — {mdl.full_sol['Time'].iloc[-1]/3600:.1f} часов")

In [ ]:
# Посмотрим на доступные колонки в результате
sol = mdl.full_sol
print(f"Всего колонок в решении: {len(sol.columns)}\n")
print("Первые 30 колонок:")
for i, col in enumerate(sol.columns[:30]):
    print(f"  {i+1:3d}. {col}")
print(f"  ... и ещё {len(sol.columns) - 30} колонок")

---
## 8. Визуализация результатов

### 8.1 Температуры

In [ ]:
hours = sol["Time"] / 3600

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Температуры основных элементов
ax = axes[0, 0]
temp_vars = {"tAir": "Воздух", "tCan": "Растения", "tFlr": "Пол", "tOut": "Снаружи"}
for var, label in temp_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("Основные температуры")
ax.set_ylabel("°C")
ax.legend()

# Температуры покрытия и экранов
ax = axes[0, 1]
cover_vars = {"tCovIn": "Покрытие (внутр.)", "tCovE": "Покрытие (внеш.)", 
              "tThScr": "Терм. экран", "tBlScr": "Светонепр. экран"}
for var, label in cover_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("Температуры покрытия и экранов")
ax.set_ylabel("°C")
ax.legend()

# Труба отопления и лампы
ax = axes[1, 0]
heat_vars = {"tPipe": "Труба отопл.", "tLamp": "Лампы (верх)", "tIntLamp": "Интерлайт"}
for var, label in heat_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("Отопление и лампы")
ax.set_ylabel("°C")
ax.legend()

# Разница температур (внутри vs снаружи)
ax = axes[1, 1]
if "tAir" in sol.columns and "tOut" in sol.columns:
    delta = sol["tAir"] - sol["tOut"]
    ax.plot(hours, delta, color="red", label="tAir - tOut")
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.fill_between(hours, delta, alpha=0.3, color="red")
    ax.set_title("Перепад температуры (внутри − снаружи)")
    ax.set_ylabel("ΔT (°C)")
    ax.legend()

for ax in axes.flat:
    ax.set_xlabel("Время (часы)")

plt.suptitle("Температурная динамика теплицы", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 8.2 CO₂ и влажность

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CO₂
ax = axes[0]
co2_vars = {"co2Air": "CO₂ в теплице", "co2Out": "CO₂ снаружи"}
for var, label in co2_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("Концентрация CO₂")
ax.set_ylabel("mg/m³")
ax.set_xlabel("Время (часы)")
ax.legend()

# Влажность
ax = axes[1]
vp_vars = {"vpAir": "Пар внутри", "vpOut": "Пар снаружи"}
for var, label in vp_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("Давление водяного пара")
ax.set_ylabel("Pa")
ax.set_xlabel("Время (часы)")
ax.legend()

plt.suptitle("CO₂ и влажность", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 8.3 Радиация и освещение

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PAR на растениях
ax = axes[0]
par_vars = {"rParSunCan": "PAR от солнца", "rParLampCan": "PAR от ламп", "rParIntLampCan": "PAR от интерлайт"}
for var, label in par_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("PAR, поглощённый растениями")
ax.set_ylabel("W/m²")
ax.legend()

# NIR
ax = axes[1]
nir_vars = {"rNirSunCan": "NIR от солнца", "rNirLampCan": "NIR от ламп"}
for var, label in nir_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label)
ax.set_title("NIR, поглощённый растениями")
ax.set_ylabel("W/m²")
ax.legend()

# parCan — итоговый PAR для фотосинтеза
ax = axes[2]
if "parCan" in sol.columns:
    ax.plot(hours, sol["parCan"], color="green", linewidth=2)
    ax.fill_between(hours, sol["parCan"], alpha=0.3, color="green")
ax.set_title("parCan — PAR для фотосинтеза")
ax.set_ylabel("µmol/m²/s")

for ax in axes:
    ax.set_xlabel("Время (часы)")

plt.suptitle("Радиация и освещение", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 8.4 Рост культуры (биомасса)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Буфер углеводов
ax = axes[0]
if "cBuf" in sol.columns:
    ax.plot(hours, sol["cBuf"], color="orange", linewidth=2)
    ax.set_title("Буфер углеводов (cBuf)")
    ax.set_ylabel("mg CH₂O / m²")

# Биомасса по органам
ax = axes[1]
biomass_vars = {"cFruit": "Плоды", "cLeaf": "Листья", "cStem": "Стебли"}
for var, label in biomass_vars.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label, linewidth=2)
ax.set_title("Биомасса по органам")
ax.set_ylabel("mg CH₂O / m²")
ax.legend()

# Индекс листовой поверхности (LAI)
ax = axes[2]
if "lai" in sol.columns:
    ax.plot(hours, sol["lai"], color="green", linewidth=2)
    ax.set_title("Индекс листовой поверхности (LAI)")
    ax.set_ylabel("m² / m²")

for ax in axes:
    ax.set_xlabel("Время (часы)")

plt.suptitle("Рост культуры", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 8.5 Система управления

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Лампы
ax = axes[0, 0]
for var, label in {"uLamp": "Лампы (вкл/выкл)", "uIntLamp": "Интерлайт"}.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label, linewidth=2)
ax.set_title("Управление освещением")
ax.set_yticks([0, 1])
ax.set_yticklabels(["ВЫКЛ", "ВКЛ"])
ax.legend()

# Вентиляция
ax = axes[0, 1]
for var, label in {"uVent": "Вентиляция (крыша)", "uSide": "Вентиляция (бок)"}.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label, linewidth=2)
# Попробуем также найти другие переменные вентиляции
vent_cols = [c for c in sol.columns if "Vent" in c or "vent" in c]
if vent_cols:
    for v in vent_cols[:3]:
        ax.plot(hours, sol[v], label=v, alpha=0.6)
ax.set_title("Вентиляция")
ax.legend()

# Отопление
ax = axes[1, 0]
if "tPipe" in sol.columns:
    ax.plot(hours, sol["tPipe"], color="red", label="Температура трубы", linewidth=2)
    ax.set_ylabel("°C")
ax.set_title("Система отопления (труба)")
ax.legend()

# Экраны
ax = axes[1, 1]
for var, label in {"uThScr": "Терм. экран", "uBlScr": "Светонепр. экран"}.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label, linewidth=2)
ax.set_title("Управление экранами")
ax.set_ylabel("Доля закрытия (0-1)")
ax.legend()

for ax in axes.flat:
    ax.set_xlabel("Время (часы)")

plt.suptitle("Система управления теплицей", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 9. Энергетический баланс: тепловые потоки

Посмотрим, как энергия распределяется между элементами теплицы:

In [ ]:
# Тепловые потоки, влияющие на tAir
heat_flows = {
    "hCanAir": "Растения → Воздух (конвекция)",
    "hAirFlr": "Воздух → Пол (конвекция)", 
    "hAirThScr": "Воздух → Терм. экран",
    "hAirOut": "Воздух → Снаружи (вентиляция)",
    "hPipeAir": "Труба → Воздух (отопление)",
    "hLampAir": "Лампы → Воздух",
    "lCanAir": "Растения → Воздух (латентное тепло)",
}

fig, ax = plt.subplots(figsize=(16, 6))

for var, label in heat_flows.items():
    if var in sol.columns:
        ax.plot(hours, sol[var], label=label, linewidth=1.5)

ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Время (часы)")
ax.set_ylabel("W/m²")
ax.set_title("Тепловые потоки, влияющие на температуру воздуха")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

---
## 10. Эксперимент: модификация параметров модели

GreenLight позволяет переопределять любые параметры через `optional_prompt`. Сравним базовую модель с модифицированной — например, изменим мощность ламп:

In [ ]:
# Эксперимент: теплица БЕЗ ламп (thetaLampMax = 0)
print("Запуск модели БЕЗ искусственного освещения...")
t0 = time_module.time()

mdl_no_lamps = GreenLight(
    optional_prompt={
        "options": {"t_end": str(n_days * 86400)},
        "params": {"thetaLampMax": {"definition": "0", "type": "const"}}
    }
)
mdl_no_lamps.load()
mdl_no_lamps.solve()
sol_no_lamps = mdl_no_lamps.full_sol
print(f"Готово за {time_module.time() - t0:.1f} сек")

# Сравнительные графики
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
hours_base = sol["Time"] / 3600
hours_mod = sol_no_lamps["Time"] / 3600

# Температура воздуха
ax = axes[0, 0]
ax.plot(hours_base, sol["tAir"], label="С лампами", linewidth=2)
ax.plot(hours_mod, sol_no_lamps["tAir"], label="Без ламп", linewidth=2, linestyle="--")
ax.set_title("Температура воздуха")
ax.set_ylabel("°C")
ax.legend()

# PAR на растениях
ax = axes[0, 1]
if "parCan" in sol.columns:
    ax.plot(hours_base, sol["parCan"], label="С лампами", linewidth=2)
    ax.plot(hours_mod, sol_no_lamps["parCan"], label="Без ламп", linewidth=2, linestyle="--")
ax.set_title("PAR для фотосинтеза (parCan)")
ax.set_ylabel("µmol/m²/s")
ax.legend()

# Буфер углеводов
ax = axes[1, 0]
if "cBuf" in sol.columns:
    ax.plot(hours_base, sol["cBuf"], label="С лампами", linewidth=2)
    ax.plot(hours_mod, sol_no_lamps["cBuf"], label="Без ламп", linewidth=2, linestyle="--")
ax.set_title("Буфер углеводов (cBuf)")
ax.set_ylabel("mg CH₂O / m²")
ax.legend()

# CO₂
ax = axes[1, 1]
if "co2Air" in sol.columns:
    ax.plot(hours_base, sol["co2Air"], label="С лампами", linewidth=2)
    ax.plot(hours_mod, sol_no_lamps["co2Air"], label="Без ламп", linewidth=2, linestyle="--")
ax.set_title("CO₂ в теплице")
ax.set_ylabel("mg/m³")
ax.legend()

for ax in axes.flat:
    ax.set_xlabel("Время (часы)")

plt.suptitle("Сравнение: С лампами vs Без ламп", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 11. Интерактивный исследователь: поиск переменных

Используйте эту ячейку для поиска любой переменной в модели по имени или описанию:

In [ ]:
def search_variables(query, all_vars):
    """Поиск переменных по имени или описанию (регистронезависимый)."""
    query_lower = query.lower()
    results = {}
    for name, v in all_vars.items():
        if (query_lower in name.lower() or 
            query_lower in v.get("description", "").lower() or
            query_lower in v.get("definition", "").lower()):
            results[name] = v
    
    if not results:
        print(f"Ничего не найдено для '{query}'")
        return
    
    print(f"Найдено {len(results)} переменных для '{query}':\n")
    for name, v in sorted(results.items()):
        print(f"  {name} [{v['type']}] ({v['unit']})")
        print(f"    Формула: {v['definition'][:100]}")
        if v["description"]:
            print(f"    Описание: {v['description'][:80]}")
        print()

# Примеры поиска — измените запрос!
search_variables("Lamp", all_vars)  # Попробуйте: "pipe", "co2", "fruit", "ventilation", "screen"

---
## 12. Интерактивный визуализатор: построение графиков любых переменных

Используйте эту ячейку для визуализации любых переменных из результатов:

In [ ]:
def plot_variables(var_names, sol_df, title="", all_vars_info=None):
    """Построить графики указанных переменных из результата симуляции.
    
    Аргументы:
        var_names: список имён переменных (строки)
        sol_df: DataFrame с результатами (mdl.full_sol)
        title: заголовок графика
        all_vars_info: словарь all_vars для добавления единиц в легенду
    """
    hours = sol_df["Time"] / 3600
    fig, ax = plt.subplots(figsize=(16, 5))
    
    for var in var_names:
        if var in sol_df.columns:
            unit = ""
            if all_vars_info and var in all_vars_info:
                unit = f" ({all_vars_info[var]['unit']})"
            ax.plot(hours, sol_df[var], label=f"{var}{unit}", linewidth=1.5)
        else:
            print(f"  Переменная '{var}' не найдена в результатах")
    
    ax.set_xlabel("Время (часы)")
    ax.set_title(title or f"Переменные: {', '.join(var_names)}")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# Пример: измените список переменных!
plot_variables(
    ["tAir", "tCan", "tPipe", "tOut"],
    sol,
    title="Температуры в теплице",
    all_vars_info=all_vars
)

---
## 13. Карта модели: визуализация потоков энергии

Схема основных потоков энергии и массы в теплице:

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

# Стиль блоков
box_style = dict(boxstyle="round,pad=0.4", facecolor="lightblue", edgecolor="navy", linewidth=2)
ext_style = dict(boxstyle="round,pad=0.4", facecolor="lightyellow", edgecolor="orange", linewidth=2)
state_style = dict(boxstyle="round,pad=0.4", facecolor="lightgreen", edgecolor="darkgreen", linewidth=2)

# Внешняя среда
ax.text(5, 9.5, "ВНЕШНЯЯ СРЕДА", ha="center", fontsize=14, fontweight="bold", 
        bbox=ext_style)
ax.text(2, 9.5, "tOut, vpOut\nco2Out, wind\niGlob", ha="center", fontsize=9, 
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

# Покрытие
ax.text(5, 8.3, "ПОКРЫТИЕ (tCovIn, tCovE)", ha="center", fontsize=11, fontweight="bold",
        bbox=state_style)

# Экраны
ax.text(2.5, 7.2, "Терм. экран\n(tThScr)", ha="center", fontsize=10, bbox=state_style)
ax.text(7.5, 7.2, "Светонепр. экран\n(tBlScr)", ha="center", fontsize=10, bbox=state_style)

# Лампы
ax.text(8.5, 6.0, "ЛАМПЫ\n(tLamp)", ha="center", fontsize=10, bbox=box_style)

# Воздух (центральный)
ax.text(5, 5.5, "ВОЗДУХ ТЕПЛИЦЫ\ntAir, vpAir, co2Air", ha="center", fontsize=12, 
        fontweight="bold", bbox=dict(boxstyle="round,pad=0.5", facecolor="lightcoral", 
                                     edgecolor="red", linewidth=2))

# Растения
ax.text(2.5, 4.0, "РАСТЕНИЯ\n(tCan, lai)", ha="center", fontsize=11, fontweight="bold",
        bbox=state_style)

# Труба
ax.text(7.5, 4.0, "ТРУБА ОТОПЛ.\n(tPipe)", ha="center", fontsize=10, bbox=box_style)

# Пол
ax.text(5, 2.5, "ПОЛ (tFlr)", ha="center", fontsize=11, bbox=state_style)

# Почва
ax.text(5, 1.2, "ПОЧВА (tSo1...tSo5)", ha="center", fontsize=10, bbox=state_style)

# Рост
ax.text(1, 2.0, "РОСТ КУЛЬТУРЫ\ncBuf → cFruit\ncLeaf, cStem", ha="center", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="palegreen", edgecolor="green", linewidth=2))

# Стрелки (потоки энергии)
arrow_style = dict(arrowstyle="->", color="navy", linewidth=1.5)
from matplotlib.patches import FancyArrowPatch

flows = [
    ((5, 9.2), (5, 8.6), "iGlob"),         # Солнце → покрытие
    ((5, 8.0), (5, 5.9), ""),                # Покрытие → воздух
    ((5, 5.1), (5, 2.8), ""),                # Воздух → пол
    ((5, 2.2), (5, 1.5), ""),                # Пол → почва
    ((3.7, 5.3), (2.5, 4.4), "hCanAir"),     # Воздух → растения
    ((7.0, 4.2), (6.2, 5.3), "hPipeAir"),    # Труба → воздух
    ((8.2, 5.7), (6.2, 5.5), "hLampAir"),    # Лампы → воздух
    ((2.5, 3.6), (1.5, 2.5), ""),            # Растения → рост
]

for start, end, label in flows:
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", color="navy", linewidth=1.5))
    if label:
        mid = ((start[0]+end[0])/2, (start[1]+end[1])/2)
        ax.text(mid[0]+0.15, mid[1]+0.1, label, fontsize=8, color="navy", fontstyle="italic")

ax.set_title("Схема потоков энергии и массы в модели GreenLight", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

---
## 14. Справочник: как модифицировать модель

### Примеры модификаций через `optional_prompt`:

```python
# Увеличить длительность симуляции до 7 дней
{"options": {"t_end": str(7 * 86400)}}

# Отключить лампы
{"params": {"thetaLampMax": {"definition": "0", "type": "const"}}}

# Изменить начальную температуру воздуха
{"params": {"tAir": {"init": "25"}}}

# Изменить мощность ламп (200 W/m²)
{"params": {"thetaLampMax": {"definition": "200", "type": "const"}}}

# Комбинировать несколько изменений
{
    "options": {"t_end": str(3 * 86400)},
    "params": {
        "thetaLampMax": {"definition": "150", "type": "const"},
        "tAir": {"init": "18"},
    }
}
```

### Структура JSON-файлов модели:
- **Секции** содержат группы переменных (температуры, радиация, управление...)
- Каждая переменная: `{"definition": "формула", "type": "тип", "unit": "единицы"}`
- Типы: `state` (ОДУ), `aux` (вычисляемая), `const`, `input`, `function`
- Файлы обрабатываются в порядке `processing_order` — позднее определение перезаписывает раннее